In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification #This loads a pretrained Transformer model with a classification head.
)

from torch.optim import AdamW

print("All imports successful!")

All imports successful!


In [ ]:
df = pd.read_csv('test.csv', encoding='latin1')

df = df[['text', 'sentiment']]

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

print("\nFirst 5 rows:")
display(df.head())

Dataset shape: (14402, 2)

Column names:
['text', 'sentiment']

Missing values:
text         0
sentiment    0
dtype: int64

Sentiment distribution:
sentiment
negative    9079
neutral     3046
positive    2277
Name: count, dtype: int64

First 5 rows:


,text,sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative


In [ ]:
df = df.dropna(subset=['text', 'sentiment']).copy()

df.reset_index(drop=True, inplace=True)

print("Dataset shape after removing missing values:", df.shape)

print("\nMissing values after cleaning:")
print(df.isnull().sum())

print("\nSentiment distribution:")
print(df['sentiment'].value_counts())

Dataset shape after removing missing values: (14402, 2)

Missing values after cleaning:
text         0
sentiment    0
dtype: int64

Sentiment distribution:
sentiment
negative    9079
neutral     3046
positive    2277
Name: count, dtype: int64


In [ ]:
# Preserve original text for DistilBERT.
# Its pretrained tokenizer handles tokenization internally.

df['clean_text'] = df['text'].astype(str).str.strip()

print("Preprocessing complete.")
print("\nSample text:")
print(df['clean_text'].iloc[0])

Preprocessing complete.

Sample text:
@VirginAmerica What @dhepburn said.


Why aren't we removing stopwords anymore?

In a traditional ML pipeline using TF-IDF, removing some stopwords can sometimes help. For DistilBERT, however, removing words or punctuation can change the context that the pretrained model uses to understand sentiment.

In [ ]:
X = df['clean_text']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.125,
    random_state=42,
    stratify=y_train
)

print("Training samples:", len(X_train_final))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

print("\nTraining distribution:")
print(y_train_final.value_counts())

print("\nValidation distribution:")
print(y_val.value_counts())

print("\nTest distribution:")
print(y_test.value_counts())

Training samples: 10080
Validation samples: 1441
Test samples: 2881

Training distribution:
sentiment
negative    6355
neutral     2132
positive    1593
Name: count, dtype: int64

Validation distribution:
sentiment
negative    908
neutral     305
positive    228
Name: count, dtype: int64

Test distribution:
sentiment
negative    1816
neutral      609
positive     456
Name: count, dtype: int64


DistilBERT.

It is a smaller, faster version of BERT and is a reasonable choice for your 3,534-example dataset and Colab environment.

In [ ]:
model_name = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

print("Tokenizer and model loaded successfully.")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer and model loaded successfully.


In [ ]:
label_map = {
    'negative': 0,
    'neutral': 1,
    'positive': 2
}

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts.tolist()
        self.labels = labels.map(label_map).tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):
        text = self.texts[index]
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            truncation=True, #If a text is longer than 128 tokens, cut it down to 128.
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),#we use squeeze because it returns [1,128] but we need only 1 dimension which is [128]
            'attention_mask': encoding['attention_mask'].squeeze(0),#his tells the Transformer which positions are actual tokens and which are padding.
            'labels': torch.tensor(label, dtype=torch.long)
        }

pt means PyTorch.

The tokenizer normally gives us Python-like arrays, but we want PyTorch tensors because our model is a PyTorch model.

In [ ]:
train_dataset = SentimentDataset(
    X_train_final,
    y_train_final,
    tokenizer
)

val_dataset = SentimentDataset(
    X_val,
    y_val,
    tokenizer
)

test_dataset = SentimentDataset(
    X_test,
    y_test,
    tokenizer
)

print("Training dataset size:", len(train_dataset))
print("Validation dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

Training dataset size: 10080
Validation dataset size: 1441
Test dataset size: 2881


In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Training batches: 630
Validation batches: 91
Test batches: 181


In [ ]:
batch = next(iter(train_loader))

print("Input IDs shape:", batch['input_ids'].shape)
print("Attention mask shape:", batch['attention_mask'].shape)
print("Labels shape:", batch['labels'].shape)

print("\nFirst example labels:", batch['labels'][:5])

Input IDs shape: torch.Size([16, 128])
Attention mask shape: torch.Size([16, 128])
Labels shape: torch.Size([16])

First example labels: tensor([1, 0, 0, 1, 0])


In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

print("Using device:", device)

Using device: cuda


In [ ]:
model = model.to(device)  #model.to(device) moves them to the GPU

print("Model moved to:", device)

Model moved to: cuda


In [ ]:
optimizer = AdamW( #AdamW is a variant of Adam that handles weight decay in a more appropriate way for many deep-learning settings, including Transformer fine-tuning.
    model.parameters(),
    lr=2e-5
)

print("Optimizer created successfully.")

Optimizer created successfully.


In [ ]:

from sklearn.utils.class_weight import compute_class_weight

# Encode training labels using the existing label map
y_train_encoded = y_train_final.map(label_map).to_numpy()

# Calculate balanced weights for all three sentiment classes
class_labels = np.array([0, 1, 2])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=class_labels,
    y=y_train_encoded
)

# Move weights to the same device as the model
class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

# Apply class weights to the loss function
loss_function = torch.nn.CrossEntropyLoss(weight=class_weights)

print("Class weights:")
for sentiment, weight in zip(
    ["negative", "neutral", "positive"],
    class_weights.cpu().tolist()
):
    print(f"{sentiment}: {weight:.4f}")


Class weights:
negative: 0.5287
neutral: 1.5760
positive: 2.1092


In [ ]:
num_epochs = 3

for epoch in range(num_epochs):

    model.train()
    total_train_loss = 0

    for batch in train_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_function(
            outputs.logits,
            labels
        )

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"- Training Loss: {average_train_loss:.4f}"
    )

Epoch 1/3 - Training Loss: 0.2491
Epoch 2/3 - Training Loss: 0.1628
Epoch 3/3 - Training Loss: 0.1087


In [ ]:
model.eval()

total_val_loss = 0
val_predictions = []
val_actuals = []

with torch.no_grad():

    for batch in val_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_function(
            outputs.logits,
            labels
        )

        total_val_loss += loss.item()

        predictions = torch.argmax( #index of largest value
            outputs.logits,#logit has[16,3] index but we need label only so dim =1
            dim=1
        )

        val_predictions.extend( #But scikit-learn's accuracy_score works with normal CPU/NumPy data.
            predictions.cpu().numpy()
        )

        val_actuals.extend(
            labels.cpu().numpy()
        )

average_val_loss = total_val_loss / len(val_loader)

val_accuracy = accuracy_score(
    val_actuals,
    val_predictions
)

print(f"Validation Loss: {average_val_loss:.4f}")
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Accuracy: {val_accuracy * 100:.2f}%")

Validation Loss: 0.8128
Validation Accuracy: 0.8411
Validation Accuracy: 84.11%


In [ ]:
print("Validation Classification Report:")

print(
    classification_report(
        val_actuals,
        val_predictions,
        target_names=['negative', 'neutral', 'positive']
    )
)

cm = confusion_matrix(
    val_actuals,
    val_predictions,
    labels=[0, 1, 2]
)

print("\nValidation Confusion Matrix:")
print(cm)

Validation Classification Report:
              precision    recall  f1-score   support

    negative       0.89      0.92      0.91       908
     neutral       0.73      0.62      0.67       305
    positive       0.75      0.82      0.79       228

    accuracy                           0.84      1441
   macro avg       0.79      0.79      0.79      1441
weighted avg       0.84      0.84      0.84      1441


Validation Confusion Matrix:
[[835  50  23]
 [ 77 190  38]
 [ 22  19 187]]


Now selecting model from best epoch


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=3
)

model = model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5
)

print("Fresh model and optimizer created.")
print("Using device:", device)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fresh model and optimizer created.
Using device: cuda


In [ ]:
num_epochs = 5

best_val_loss = float('inf')
best_model_state = None

patience = 2
epochs_without_improvement = 0

print("Training configuration:")
print("Maximum epochs:", num_epochs)
print("Early stopping patience:", patience)

Training configuration:
Maximum epochs: 5
Early stopping patience: 2


In [ ]:
for epoch in range(num_epochs):

    # -------------------------
    # Training
    # -------------------------
    model.train()
    total_train_loss = 0

    for batch in train_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_function(
            outputs.logits,
            labels
        )

        loss.backward()
        optimizer.step()

        total_train_loss += loss.item()

    average_train_loss = total_train_loss / len(train_loader)

    # -------------------------
    # Validation
    # -------------------------
    model.eval()

    total_val_loss = 0
    val_predictions = []
    val_actuals = []

    with torch.no_grad():

        for batch in val_loader:

            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            loss = loss_function(
                outputs.logits,
                labels
            )

            total_val_loss += loss.item()

            predictions = torch.argmax(
                outputs.logits,
                dim=1
            )

            val_predictions.extend(
                predictions.cpu().numpy()
            )

            val_actuals.extend(
                labels.cpu().numpy()
            )

    average_val_loss = total_val_loss / len(val_loader)

    val_accuracy = accuracy_score(
        val_actuals,
        val_predictions
    )

    # -------------------------
    # Print results
    # -------------------------
    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"- Train Loss: {average_train_loss:.4f} "
        f"- Val Loss: {average_val_loss:.4f} "
        f"- Val Accuracy: {val_accuracy:.4f}"
    )

    # -------------------------
    # Check for improvement
    # -------------------------
    if average_val_loss < best_val_loss:

        best_val_loss = average_val_loss

        best_model_state = {
            key: value.cpu().clone()
            for key, value in model.state_dict().items()
        }

        epochs_without_improvement = 0

        print("  → Validation loss improved. Best model saved.")

    else:

        epochs_without_improvement += 1

        print(
            f"  → No improvement "
            f"({epochs_without_improvement}/{patience})"
        )

        if epochs_without_improvement >= patience:

            print("  → Early stopping triggered.")
            break

Epoch 1/5 - Train Loss: 0.6221 - Val Loss: 0.4990 - Val Accuracy: 0.8085
  → Validation loss improved. Best model saved.
Epoch 2/5 - Train Loss: 0.3907 - Val Loss: 0.4852 - Val Accuracy: 0.8369
  → Validation loss improved. Best model saved.
Epoch 3/5 - Train Loss: 0.2501 - Val Loss: 0.5335 - Val Accuracy: 0.8258
  → No improvement (1/2)
Epoch 4/5 - Train Loss: 0.1587 - Val Loss: 0.7119 - Val Accuracy: 0.8480
  → No improvement (2/2)
  → Early stopping triggered.


In [ ]:
model.load_state_dict(best_model_state)

model = model.to(device)

print(f"Best validation loss: {best_val_loss:.4f}")
print("Best model restored successfully.")

Best validation loss: 0.4852
Best model restored successfully.


In [ ]:
model.eval()

total_test_loss = 0
test_predictions = []
test_actuals = []

with torch.no_grad():

    for batch in test_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        loss = loss_function(
            outputs.logits,
            labels
        )

        total_test_loss += loss.item()

        predictions = torch.argmax(
            outputs.logits,
            dim=1
        )

        test_predictions.extend(
            predictions.cpu().numpy()
        )

        test_actuals.extend(
            labels.cpu().numpy()
        )

average_test_loss = total_test_loss / len(test_loader)

test_accuracy = accuracy_score(
    test_actuals,
    test_predictions
)

print(f"Test Loss: {average_test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

Test Loss: 0.4813
Test Accuracy: 0.8268
Test Accuracy: 82.68%


In [ ]:
print("Test Classification Report:")

print(
    classification_report(
        test_actuals,
        test_predictions,
        target_names=['negative', 'neutral', 'positive']
    )
)

cm = confusion_matrix(
    test_actuals,
    test_predictions,
    labels=[0, 1, 2]
)

print("\nTest Confusion Matrix:")
print(cm)

Test Classification Report:
              precision    recall  f1-score   support

    negative       0.91      0.86      0.89      1816
     neutral       0.68      0.67      0.68       609
    positive       0.72      0.88      0.79       456

    accuracy                           0.83      2881
   macro avg       0.77      0.81      0.79      2881
weighted avg       0.83      0.83      0.83      2881


Test Confusion Matrix:
[[1569  156   91]
 [ 129  411   69]
 [  18   36  402]]


## Conclusion: DistilBERT Sentiment Classification

In this experiment, I fine-tuned a pretrained DistilBERT model to classify tweets into negative, neutral, and positive sentiment categories.

The model achieved **82.68% test accuracy**, a **macro F1-score of 0.79**, and a **weighted F1-score of 0.83**. Training used class-weighted cross-entropy loss to account for the imbalance between sentiment classes. Early stopping based on validation loss helped select the best model checkpoint.

The model performed best on negative sentiment, with an F1-score of 0.89, followed by positive sentiment at 0.79. Neutral sentiment was more difficult to classify, with an F1-score of 0.68. This suggests that distinguishing neutral language from clearly positive or negative language remains a key challenge.

Overall, DistilBERT achieved stronger results than the TF-IDF-based Logistic Regression and PyTorch neural network experiments. Further improvements could focus on neutral sentiment classification, error analysis, and systematic hyperparameter tuning.